## Город Ташкент

In [4]:
import pandas as pd
import os
import pandas as pd
import psycopg2
import requests
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# Функция для получения актуального курса USD/UZS
def get_usd_to_uzs():
    url = "https://cbu.uz/ru/arkhiv-kursov-valyut/json/"
    try:
        response = requests.get(url)
        data = response.json()
        for currency in data:
            if currency["Ccy"] == "USD":
                return float(currency["Rate"])
    except Exception as e:
        print("Ошибка получения курса:", e)
        return 12895  # запасное значение

# Получаем текущий курс валют
usd_to_uzs = get_usd_to_uzs()
print("Текущий курс USD/UZS:", usd_to_uzs)

# Подключение к PostgreSQL
conn = psycopg2.connect(
    dbname="postgres",
    user="gulimoh",
    password="postgres",  # Укажи свой пароль
    host="127.0.0.1",
    port="5432"
)

# SQL-запрос для загрузки данных
query = f"""
WITH converted_prices AS (
    SELECT *,
           CASE
               WHEN currency = 'UYE' THEN price
               ELSE price / {usd_to_uzs}
           END AS price_uye,
           CASE
               WHEN currency = 'UYE' THEN price * {usd_to_uzs}
               ELSE price
           END AS price_uz
    FROM apartments
    WHERE regionname = 'Ташкентская область'
      AND cityname = 'Ташкент'
      AND createdtime >= '2024-02-01'
      AND total_area >= 18
),
calculated_prices AS (
    SELECT *,
           CASE
               WHEN price_uye <= 5000 THEN price_uye
               ELSE price_uye / total_area
           END AS price_per_m2_uye,
           CASE
               WHEN price_uz <= 5000 * {usd_to_uzs} THEN price_uz
               ELSE price_uz / total_area
           END AS price_per_m2_uz
    FROM converted_prices
),
filtered_data AS (
    SELECT *,
           CASE
               WHEN number_of_rooms = 1 THEN total_area BETWEEN 18 AND 70
               WHEN number_of_rooms = 2 THEN total_area BETWEEN 25 AND 100
               WHEN number_of_rooms = 3 THEN total_area BETWEEN 30 AND 150
               WHEN number_of_rooms = 4 THEN total_area BETWEEN 50 AND 150
               ELSE total_area BETWEEN 80 AND 400
           END AS valid_area
    FROM calculated_prices
),
cleaned_floors AS (
    SELECT *,
           LEAST(floor, 25) AS floor_limited,
           LEAST(total_floors, 25) AS total_floors_limited
    FROM filtered_data
),
final_filtered AS (
    SELECT * FROM cleaned_floors
    WHERE valid_area
      AND price_per_m2_uye BETWEEN 700 AND 5000
)
SELECT number_of_rooms, total_area, floor_limited AS floor, total_floors_limited AS total_floors,
       furnished, repairs, comission, wc, house_type, isbusiness, ishighlighted, ispromoted, negotiable,
       districtname, layout, ceiling_height, price_per_m2_uye, url, description
FROM final_filtered;
"""

# Загружаем данные
df = pd.read_sql(query, conn)
conn.close()

# Преобразуем ceiling_height в числовой формат
df["ceiling_height"] = pd.to_numeric(df["ceiling_height"], errors='coerce')


pd.set_option('display.max_rows', None)

# Загрузка данных

# # Удаляем объявления с deal_type == "rent"
# df = df[df['deal_type'] != 'rent']  # оставляем только не-rent

# # # Сохраняем результат в тот же файл
# df.to_csv(input_csv, index=False, encoding="utf-8")
#
# # Фильтруем только те, у которых region = Tashkent или unknown
# df = telegram_all_df[telegram_all_df['district'].isin(["unknown"])]

#
# district_map = {
#     "апартамент": "apartment",
#     # "unknown": "house"
# }
# # Заменяем district по словарю
# df["type"] = df["type"].replace(district_map)
# df["type"] = df["type"].astype(str).str.strip()




# # # ===== Замена rent -> sale =====
# df["deal_type"] = df["deal_type"].replace("rent", "sale")

# df.to_csv(input_csv, index=False, encoding="utf-8")

# Список интересующих столбцов
columns_of_interest = [
    # 'type',
    # 'deal_type',
    # 'district',
    # 'landmark',
    # 'condition',
    # 'furnished',
    'repairs',
    # 'building_material',
    # 'region'
    # 'price'
    # 'address'
]

# Выводим уникальные значения
for column in columns_of_interest:
    print(f"Уникальные значения в колонке '{column}':")
    print(df[column].value_counts(dropna=False))
    print("\n" + "="*50 + "\n")

# print(df_cleaned["deal_type"].value_counts(dropna=False))

Текущий курс USD/UZS: 12920.16


/var/folders/q0/042269_x4qggn_0htwwx67qr0000gq/T/ipykernel_4145/3494964232.py:95: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Уникальные значения в колонке 'repairs':
Евроремонт              28407
Средний                 16525
Авторский проект         8052
None                     6342
Черновая отделка         4579
Требует ремонта          4185
Предчистовая отделка     1691
Name: repairs, dtype: int64




In [3]:
import os
import pandas as pd
import psycopg2
import requests
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# Функция для получения актуального курса USD/UZS
def get_usd_to_uzs():
    url = "https://cbu.uz/ru/arkhiv-kursov-valyut/json/"
    try:
        response = requests.get(url)
        data = response.json()
        for currency in data:
            if currency["Ccy"] == "USD":
                return float(currency["Rate"])
    except Exception as e:
        print("Ошибка получения курса:", e)
        return 12895  # запасное значение

# Получаем текущий курс валют
usd_to_uzs = get_usd_to_uzs()
print("Текущий курс USD/UZS:", usd_to_uzs)

# Подключение к PostgreSQL
conn = psycopg2.connect(
    dbname="postgres",
    user="gulimoh",
    password="postgres",  # Укажи свой пароль
    host="127.0.0.1",
    port="5432"
)

# SQL-запрос для загрузки данных
query = f"""
WITH converted_prices AS (
    SELECT *,
           CASE
               WHEN currency = 'UYE' THEN price
               ELSE price / {usd_to_uzs}
           END AS price_uye,
           CASE
               WHEN currency = 'UYE' THEN price * {usd_to_uzs}
               ELSE price
           END AS price_uz
    FROM apartments
    WHERE regionname = 'Ташкентская область'
      AND cityname = 'Ташкент'
      AND createdtime >= '2024-02-01'
      AND total_area >= 18
),
calculated_prices AS (
    SELECT *,
           CASE
               WHEN price_uye <= 5000 THEN price_uye
               ELSE price_uye / total_area
           END AS price_per_m2_uye,
           CASE
               WHEN price_uz <= 5000 * {usd_to_uzs} THEN price_uz
               ELSE price_uz / total_area
           END AS price_per_m2_uz
    FROM converted_prices
),
filtered_data AS (
    SELECT *,
           CASE
               WHEN number_of_rooms = 1 THEN total_area BETWEEN 18 AND 70
               WHEN number_of_rooms = 2 THEN total_area BETWEEN 25 AND 100
               WHEN number_of_rooms = 3 THEN total_area BETWEEN 30 AND 150
               WHEN number_of_rooms = 4 THEN total_area BETWEEN 50 AND 150
               ELSE total_area BETWEEN 80 AND 400
           END AS valid_area
    FROM calculated_prices
),
cleaned_floors AS (
    SELECT *,
           LEAST(floor, 25) AS floor_limited,
           LEAST(total_floors, 25) AS total_floors_limited
    FROM filtered_data
),
final_filtered AS (
    SELECT * FROM cleaned_floors
    WHERE valid_area
      AND price_per_m2_uye BETWEEN 700 AND 5000
)
SELECT number_of_rooms, total_area, floor_limited AS floor, total_floors_limited AS total_floors,
       furnished, repairs, comission, wc, house_type, isbusiness, ishighlighted, ispromoted, negotiable,
       districtname, layout, ceiling_height, price_per_m2_uye, url, description
FROM final_filtered;
"""

# Загружаем данные
df = pd.read_sql(query, conn)
conn.close()

# Преобразуем ceiling_height в числовой формат
df["ceiling_height"] = pd.to_numeric(df["ceiling_height"], errors='coerce')

# Удаляем столбцы с пропусками ≥ 70%
df = df.loc[:, df.isnull().mean() < 0.7]

# Обрабатываем категориальные признаки (One-Hot Encoding)
df = pd.get_dummies(df, columns=["furnished", "repairs", "wc", "house_type", "districtname", "layout"], drop_first=True)

# Заполняем пропуски средним значением для числовых признаков
df.fillna(df.mean(), inplace=True)




# Создаём пустой DataFrame для результатов
results = pd.DataFrame(columns=["Регион", "Количество комнат", "MAE", "MAPE (%)", "Ошибки >10% (%)"])

# Список уникальных значений комнат
room_categories = [1, 2, 3, 4, 5]

for rooms in room_categories:
    # print(f"\nОбучение модели для {rooms}-комнатных квартир...")

    if rooms == 5:
        df_filtered = df[df["number_of_rooms"] >= 5]  # 5 и больше
    else:
        df_filtered = df[df["number_of_rooms"] == rooms]

    if df_filtered.empty:
        print(f"Пропускаем {rooms}-комнатные: нет данных.")
        continue

    # Разделение данных на train/test (80/20)
    X = df_filtered.drop(columns=["price_per_m2_uye", "number_of_rooms"])
    y = df_filtered["price_per_m2_uye"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

#     # Обучение модели XGBoost
#     apartments_model = xgb.XGBRegressor(
#         objective="reg:squarederror",
#         n_estimators=100,
#         learning_rate=0.1,
#         random_state=42,
#         n_jobs=1
#     )
#     apartments_model.fit(X_train, y_train)
#
#     # Предсказание и метрики
#     y_pred = apartments_model.predict(X_test)
#     mae = mean_absolute_error(y_test, y_pred)
#     mape = mean_absolute_percentage_error(y_test, y_pred)
#     error_10_percent = (abs(y_test - y_pred) / y_test > 0.1).mean()
#
#     # Добавляем результаты в DataFrame
#     results = results.append({
#         "Регион": "Ташкент",
#         "Количество комнат": rooms,
#         "MAE": round(mae, 2),
#         "MAPE (%)": round(mape * 100, 2),
#         "Ошибки >10% (%)": round(error_10_percent * 100, 2)
#     }, ignore_index=True)
#
# # Вывод таблицы
# results


Текущий курс USD/UZS: 12933.16


/var/folders/q0/042269_x4qggn_0htwwx67qr0000gq/T/ipykernel_4145/3247042100.py:94: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
/var/folders/q0/042269_x4qggn_0htwwx67qr0000gq/T/ipykernel_4145/3247042100.py:107: FutureWarning: The default value of numeric_only in DataFrame.mean is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  df.fillna(df.mean(), inplace=True)


In [11]:
import os
import pandas as pd
import psycopg2
import requests
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# Функция для получения актуального курса USD/UZS
def get_usd_to_uzs():
    url = "https://cbu.uz/ru/arkhiv-kursov-valyut/json/"
    try:
        response = requests.get(url)
        data = response.json()
        for currency in data:
            if currency["Ccy"] == "USD":
                return float(currency["Rate"])
    except Exception as e:
        print("Ошибка получения курса:", e)
        return 12895  # запасное значение

# Получаем текущий курс валют
usd_to_uzs = get_usd_to_uzs()
print("Текущий курс USD/UZS:", usd_to_uzs)

# Подключение к PostgreSQL
conn = psycopg2.connect(
    dbname="postgres",
    user="gulimoh",
    password="postgres",  # Укажи свой пароль
    host="127.0.0.1",
    port="5432"
)

# SQL-запрос для загрузки данных
query = f"""
WITH converted_prices AS (
    SELECT *,
           CASE
               WHEN currency = 'UYE' THEN price
               ELSE price / {usd_to_uzs}
           END AS price_uye,
           CASE
               WHEN currency = 'UYE' THEN price * {usd_to_uzs}
               ELSE price
           END AS price_uz
    FROM apartments
    WHERE regionname = 'Ташкентская область'
      AND cityname = 'Ташкент'
      AND createdtime >= '2024-02-01'
      AND total_area >= 18
),
calculated_prices AS (
    SELECT *,
           CASE
               WHEN price_uye <= 5000 THEN price_uye
               ELSE price_uye / total_area
           END AS price_per_m2_uye,
           CASE
               WHEN price_uz <= 5000 * {usd_to_uzs} THEN price_uz
               ELSE price_uz / total_area
           END AS price_per_m2_uz
    FROM converted_prices
),
filtered_data AS (
    SELECT *,
           CASE
               WHEN number_of_rooms = 1 THEN total_area BETWEEN 18 AND 70
               WHEN number_of_rooms = 2 THEN total_area BETWEEN 25 AND 100
               WHEN number_of_rooms = 3 THEN total_area BETWEEN 30 AND 150
               WHEN number_of_rooms = 4 THEN total_area BETWEEN 50 AND 150
               ELSE total_area BETWEEN 80 AND 400
           END AS valid_area
    FROM calculated_prices
),
cleaned_floors AS (
    SELECT *,
           LEAST(floor, 25) AS floor_limited,
           LEAST(total_floors, 25) AS total_floors_limited
    FROM filtered_data
),
final_filtered AS (
    SELECT * FROM cleaned_floors
    WHERE valid_area
      AND price_per_m2_uye BETWEEN 700 AND 5000
)
SELECT number_of_rooms, total_area, floor_limited AS floor, total_floors_limited AS total_floors,
       furnished, repairs, comission, wc, house_type, isbusiness, ishighlighted, ispromoted, negotiable,
       districtname, layout, ceiling_height, price_per_m2_uye
FROM final_filtered;
"""

# Загружаем данные
df = pd.read_sql(query, conn)
conn.close()

# Преобразуем ceiling_height в числовой формат
df["ceiling_height"] = pd.to_numeric(df["ceiling_height"], errors='coerce')

# Удаляем столбцы с пропусками ≥ 70%
df = df.loc[:, df.isnull().mean() < 0.7]

# Обрабатываем категориальные признаки (One-Hot Encoding)
# df = pd.get_dummies(df, columns=["furnished", "repairs", "wc", "house_type", "districtname", "layout"], drop_first=True)

# Заполняем пропуски средним значением для числовых признаков
# df.fillna(df.mean(), inplace=True)

from sklearn.preprocessing import OneHotEncoder
import joblib

# Список категориальных колонок
categorical_cols = ["furnished", "repairs", "wc", "house_type", "districtname", "layout"]

# Убираем строки с NaN в категориальных колонках — либо заполнишь заранее
df = df.dropna(subset=categorical_cols)

# Отдельно категориальные и числовые
X_cat = df[categorical_cols].astype(str)
X_num = df.drop(columns=categorical_cols + ["price_per_m2_uye", "number_of_rooms"])

# Инициализируем OneHotEncoder
encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
X_cat_encoded = encoder.fit_transform(X_cat)

# Сохраняем энкодер
joblib.dump(encoder, "encoder.joblib")

# Получаем названия новых колонок
encoded_feature_names = encoder.get_feature_names_out(categorical_cols)
X_cat_encoded_df = pd.DataFrame(X_cat_encoded, columns=encoded_feature_names, index=df.index)

# Объединяем с числовыми
X_full = pd.concat([X_num, X_cat_encoded_df], axis=1)
y_full = df["price_per_m2_uye"]
rooms_full = df["number_of_rooms"]


# Создаём пустой DataFrame для результатов
results = pd.DataFrame(columns=["Регион", "Количество комнат", "MAE", "MAPE (%)", "Ошибки >10% (%)"])

# Список уникальных значений комнат
room_categories = [1, 2, 3, 4, 5]

for rooms in room_categories:
    # print(f"\nОбучение модели для {rooms}-комнатных квартир...")

    if rooms == 5:
        df_filtered = df[df["number_of_rooms"] >= 5]  # 5 и больше
    else:
        df_filtered = df[df["number_of_rooms"] == rooms]

    if df_filtered.empty:
        print(f"Пропускаем {rooms}-комнатные: нет данных.")
        continue

    # Разделение данных на train/test (80/20)
    X = X_full.loc[rooms_full == rooms]
    y = y_full.loc[rooms_full == rooms]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

    # Обучение модели XGBoost
    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=100,
        learning_rate=0.1,
        random_state=42,
        n_jobs=1
    )
    model.fit(X_train, y_train)

    # Предсказание и метрики
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    error_10_percent = (abs(y_test - y_pred) / y_test > 0.1).mean()

    # Добавляем результаты в DataFrame
    results = results.append({
        "_": "olx",
        "Регион": "Ташкент",
        "Количество комнат": rooms,
        "MAE": round(mae, 2),
        "MAPE (%)": round(mape * 100, 2),
        "Ошибки >10% (%)": round(error_10_percent * 100, 2)
    }, ignore_index=True)
model.save_model("final_model.json")
# Вывод таблицы
results


Текущий курс USD/UZS: 12924.92


/var/folders/q0/042269_x4qggn_0htwwx67qr0000gq/T/ipykernel_4423/1491702256.py:94: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
/Users/gulimoh/Desktop/olx-scraper-master/env/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/var/folders/q0/042269_x4qggn_0htwwx67qr0000gq/T/ipykernel_4423/1491702256.py:180: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append({
/var/folders/q0/042269_x4qggn_0htwwx67qr0000gq/T/ipykernel_4423/1491702256.py:180: FutureWarning: The frame.append method is deprecated

,Регион,Количество комнат,MAE,MAPE (%),Ошибки >10% (%),_
0,Ташкент,1,153.83,11.22,41.94,olx
1,Ташкент,2,177.69,12.92,45.60,olx
2,Ташкент,3,197.48,14.44,50.92,olx
3,Ташкент,4,198.82,15.71,52.52,olx
4,Ташкент,5,180.98,15.86,54.76,olx


In [12]:
# apartment_price_pipeline.py

import os
import pandas as pd
import psycopg2
import requests
import joblib
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import xgboost as xgb

# Получение актуального курса

def get_usd_to_uzs():
    url = "https://cbu.uz/ru/arkhiv-kursov-valyut/json/"
    try:
        response = requests.get(url)
        data = response.json()
        for currency in data:
            if currency["Ccy"] == "USD":
                return float(currency["Rate"])
    except Exception as e:
        print("Ошибка получения курса:", e)
        return 12895

usd_to_uzs = get_usd_to_uzs()
print("Текущий курс USD/UZS:", usd_to_uzs)

# Подключение к БД
conn = psycopg2.connect(
    dbname="postgres",
    user="gulimoh",
    password="postgres",
    host="127.0.0.1",
    port="5432"
)

# SQL-запрос
query = f"""
WITH converted_prices AS (
    SELECT *,
           CASE WHEN currency = 'UYE' THEN price ELSE price / {usd_to_uzs} END AS price_uye,
           CASE WHEN currency = 'UYE' THEN price * {usd_to_uzs} ELSE price END AS price_uz
    FROM apartments
    WHERE regionname = 'Ташкентская область'
      AND cityname = 'Ташкент'
      AND createdtime >= '2024-02-01'
      AND total_area >= 18
),
calculated_prices AS (
    SELECT *,
           CASE WHEN price_uye <= 5000 THEN price_uye ELSE price_uye / total_area END AS price_per_m2_uye,
           CASE WHEN price_uz <= 5000 * {usd_to_uzs} THEN price_uz ELSE price_uz / total_area END AS price_per_m2_uz
    FROM converted_prices
),
filtered_data AS (
    SELECT *,
           CASE
               WHEN number_of_rooms = 1 THEN total_area BETWEEN 18 AND 70
               WHEN number_of_rooms = 2 THEN total_area BETWEEN 25 AND 100
               WHEN number_of_rooms = 3 THEN total_area BETWEEN 30 AND 150
               WHEN number_of_rooms = 4 THEN total_area BETWEEN 50 AND 150
               ELSE total_area BETWEEN 80 AND 400
           END AS valid_area
    FROM calculated_prices
),
cleaned_floors AS (
    SELECT *,
           LEAST(floor, 25) AS floor_limited,
           LEAST(total_floors, 25) AS total_floors_limited
    FROM filtered_data
),
final_filtered AS (
    SELECT * FROM cleaned_floors
    WHERE valid_area AND price_per_m2_uye BETWEEN 700 AND 5000
)
SELECT number_of_rooms, total_area, floor_limited AS floor, total_floors_limited AS total_floors,
       furnished, repairs, comission, wc, house_type, isbusiness, ishighlighted, ispromoted, negotiable,
       districtname, layout, ceiling_height, price_per_m2_uye
FROM final_filtered;
"""

# Загрузка данных
print("Загружаем данные...")
df = pd.read_sql(query, conn)
conn.close()

# Приведение типов
print("Предобработка данных...")
df["ceiling_height"] = pd.to_numeric(df["ceiling_height"], errors='coerce')
df = df.loc[:, df.isnull().mean() < 0.7]  # удаление сильно пропущенных

categorical_cols = ["furnished", "repairs", "wc", "house_type", "districtname", "layout"]
df = df.dropna(subset=categorical_cols)
X_cat = df[categorical_cols].astype(str)
X_num = df.drop(columns=categorical_cols + ["price_per_m2_uye", "number_of_rooms"])

# OneHotEncoder
encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
X_cat_encoded = encoder.fit_transform(X_cat)
joblib.dump(encoder, "encoder.joblib")

X_cat_encoded_df = pd.DataFrame(X_cat_encoded, columns=encoder.get_feature_names_out(categorical_cols), index=df.index)
X_full = pd.concat([X_num, X_cat_encoded_df], axis=1)
y_full = df["price_per_m2_uye"]
rooms_full = df["number_of_rooms"]

# Сохранение названий колонок
with open("column_dtypes.json", "w") as f:
    json.dump(list(X_full.columns), f)

# Модель по комнатам
results = pd.DataFrame(columns=["Регион", "Количество комнат", "MAE", "MAPE (%)", "Ошибки >10% (%)"])
room_categories = [1, 2, 3, 4, 5]

for rooms in room_categories:
    if rooms == 5:
        X = X_full.loc[rooms_full >= 5]
        y = y_full.loc[rooms_full >= 5]
    else:
        X = X_full.loc[rooms_full == rooms]
        y = y_full.loc[rooms_full == rooms]

    if X.empty:
        print(f"Пропускаем {rooms}-комнатные: нет данных.")
        continue

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=100,
        learning_rate=0.1,
        random_state=42,
        n_jobs=1
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    error_10_percent = (abs(y_test - y_pred) / y_test > 0.1).mean()

    results = results.append({
        "Регион": "Ташкент",
        "Количество комнат": rooms,
        "MAE": round(mae, 2),
        "MAPE (%)": round(mape * 100, 2),
        "Ошибки >10% (%)": round(error_10_percent * 100, 2)
    }, ignore_index=True)

# Сохраняем последнюю обученную модель
model.save_model("final_model.json")
print("Модель и энкодер сохранены.")
print(results)


Текущий курс USD/UZS: 12924.92
Загружаем данные...


/var/folders/q0/042269_x4qggn_0htwwx67qr0000gq/T/ipykernel_4423/498707741.py:87: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Предобработка данных...


/Users/gulimoh/Desktop/olx-scraper-master/env/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/var/folders/q0/042269_x4qggn_0htwwx67qr0000gq/T/ipykernel_4423/498707741.py:145: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append({
/var/folders/q0/042269_x4qggn_0htwwx67qr0000gq/T/ipykernel_4423/498707741.py:145: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append({
/var/folders/q0/042269_x4qggn_0htwwx67qr0000gq/T/ipykernel_4423/498707741.py:145: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat i

Модель и энкодер сохранены.
    Регион Количество комнат     MAE  MAPE (%)  Ошибки >10% (%)
0  Ташкент                 1  154.93     11.63            42.62
1  Ташкент                 2  175.69     12.45            44.30
2  Ташкент                 3  204.23     15.11            51.84
3  Ташкент                 4  201.39     15.13            53.14
4  Ташкент                 5  270.20     20.54            54.20


/var/folders/q0/042269_x4qggn_0htwwx67qr0000gq/T/ipykernel_4423/498707741.py:145: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append({
/var/folders/q0/042269_x4qggn_0htwwx67qr0000gq/T/ipykernel_4423/498707741.py:145: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  results = results.append({


## Ташкентская область

In [18]:
import os
import pandas as pd
import psycopg2
import requests
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)


# Функция для получения актуального курса USD/UZS
def get_usd_to_uzs():
    url = "https://cbu.uz/ru/arkhiv-kursov-valyut/json/"
    try:
        response = requests.get(url)
        data = response.json()
        for currency in data:
            if currency["Ccy"] == "USD":
                return float(currency["Rate"])
    except Exception as e:
        print("Ошибка получения курса:", e)
        return 12895  # запасное значение

# Получаем текущий курс валют
usd_to_uzs = get_usd_to_uzs()
print("Текущий курс USD/UZS:", usd_to_uzs)

# Подключение к PostgreSQL
conn = psycopg2.connect(
    dbname="postgres",
    user="gulimoh",
    password="postgres",  # Укажи свой пароль
    host="127.0.0.1",
    port="5432"
)

# SQL-запрос для загрузки данных
query = f"""
WITH converted_prices AS (
    SELECT *,
           CASE
               WHEN currency = 'UYE' THEN price
               ELSE price / {usd_to_uzs}
           END AS price_uye,
           CASE
               WHEN currency = 'UYE' THEN price * {usd_to_uzs}
               ELSE price
           END AS price_uz
    FROM apartments
    WHERE regionname = 'Ташкентская область'
      AND cityname <> 'Ташкент'
      AND createdtime >= '2024-02-01'
      AND total_area >= 18
),
calculated_prices AS (
    SELECT *,
           CASE
               WHEN price_uye <= 5000 THEN price_uye
               ELSE price_uye / total_area
           END AS price_per_m2_uye,
           CASE
               WHEN price_uz <= 5000 * {usd_to_uzs} THEN price_uz
               ELSE price_uz / total_area
           END AS price_per_m2_uz
    FROM converted_prices
),
filtered_data AS (
    SELECT *,
           CASE
               WHEN number_of_rooms = 1 THEN total_area BETWEEN 18 AND 70
               WHEN number_of_rooms = 2 THEN total_area BETWEEN 25 AND 100
               WHEN number_of_rooms = 3 THEN total_area BETWEEN 30 AND 150
               WHEN number_of_rooms = 4 THEN total_area BETWEEN 50 AND 150
               ELSE total_area BETWEEN 80 AND 400
           END AS valid_area
    FROM calculated_prices
),
cleaned_floors AS (
    SELECT *,
           LEAST(floor, 25) AS floor_limited,
           LEAST(total_floors, 25) AS total_floors_limited
    FROM filtered_data
),
final_filtered AS (
    SELECT * FROM cleaned_floors
    WHERE valid_area
      AND price_per_m2_uye BETWEEN 300 AND 3000
)
SELECT number_of_rooms, total_area, floor_limited AS floor, total_floors_limited AS total_floors,
       furnished, repairs, comission, wc, house_type, isbusiness, ishighlighted, ispromoted, negotiable,
        layout, cityname,  ceiling_height, price_per_m2_uye
FROM final_filtered;
"""

# Загружаем данные
df = pd.read_sql(query, conn)
conn.close()

# Преобразуем ceiling_height в числовой формат
df["ceiling_height"] = pd.to_numeric(df["ceiling_height"], errors='coerce')

# Удаляем столбцы с пропусками ≥ 70%
df = df.loc[:, df.isnull().mean() < 0.7]

# Обрабатываем категориальные признаки (One-Hot Encoding)
df = pd.get_dummies(df, columns=["furnished", "repairs", "wc", "house_type", "cityname", "layout"], drop_first=True)

# Заполняем пропуски средним значением для числовых признаков
df.fillna(df.mean(), inplace=True)




# Создаём пустой DataFrame для результатов
results = pd.DataFrame(columns=["Регион", "Количество комнат", "MAE", "MAPE (%)", "Ошибки >10% (%)"])

# Список уникальных значений комнат
room_categories = [1, 2, 3, 4, 5]

for rooms in room_categories:
    # print(f"\nОбучение модели для {rooms}-комнатных квартир...")

    if rooms == 5:
        df_filtered = df[df["number_of_rooms"] >= 5]  # 5 и больше
    else:
        df_filtered = df[df["number_of_rooms"] == rooms]

    if df_filtered.empty:
        print(f"Пропускаем {rooms}-комнатные: нет данных.")
        continue

    # Разделение данных на train/test (80/20)
    X = df_filtered.drop(columns=["price_per_m2_uye", "number_of_rooms"])
    y = df_filtered["price_per_m2_uye"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

    # Обучение модели XGBoost
    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=100,
        learning_rate=0.1,
        random_state=42,
        n_jobs=1
    )
    model.fit(X_train, y_train)

    # Предсказание и метрики
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    error_10_percent = (abs(y_test - y_pred) / y_test > 0.1).mean()

    # Добавляем результаты в DataFrame
    results = results.append({
        "Регион": "Ташкентская область",
        "Количество комнат": rooms,
        "MAE": round(mae, 2),
        "MAPE (%)": round(mape * 100, 2),
        "Ошибки >10% (%)": round(error_10_percent * 100, 2)
    }, ignore_index=True)

# Вывод таблицы
results



Текущий курс USD/UZS: 12879.95


/var/folders/q0/042269_x4qggn_0htwwx67qr0000gq/T/ipykernel_45694/3917841489.py:97: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,Регион,Количество комнат,MAE,MAPE (%),Ошибки >10% (%)
0,Ташкентская область,1,110.16,14.83,41.09
1,Ташкентская область,2,138.93,17.32,52.17
2,Ташкентская область,3,128.30,16.05,51.22
3,Ташкентская область,4,190.10,18.57,57.14
4,Ташкентская область,5,256.22,41.53,61.11


##  Бухарская область, Самаркандская область, Навоийская область, Ферганская область

In [17]:
import os
import pandas as pd
import psycopg2
import requests
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

# Функция для получения актуального курса USD/UZS
def get_usd_to_uzs():
    url = "https://cbu.uz/ru/arkhiv-kursov-valyut/json/"
    try:
        response = requests.get(url)
        data = response.json()
        for currency in data:
            if currency["Ccy"] == "USD":
                return float(currency["Rate"])
    except Exception as e:
        print("Ошибка получения курса:", e)
        return 12895  # запасное значение

# Получаем текущий курс валют
usd_to_uzs = get_usd_to_uzs()
print("Текущий курс USD/UZS:", usd_to_uzs)

# Подключение к PostgreSQL
conn = psycopg2.connect(
    dbname="postgres",
    user="gulimoh",
    password="postgres",  # Укажи свой пароль
    host="127.0.0.1",
    port="5432"
)

# Границы выбросов по областям
outlier_bounds = {
    "Бухарская область": (150, 1200),
    "Самаркандская область": (300, 5000),
    "Навоийская область": (200, 1200),
    "Ферганская область": (200, 1200)
}

# Загружаем данные по всем нужным областям
regions = "', '".join(outlier_bounds.keys())

query = f"""
WITH converted_prices AS (
    SELECT *,
           CASE
               WHEN currency = 'UYE' THEN price
               ELSE price / {usd_to_uzs}
           END AS price_uye,
           CASE
               WHEN currency = 'UYE' THEN price * {usd_to_uzs}
               ELSE price
           END AS price_uz
    FROM apartments
    WHERE regionname IN ('{regions}')
      AND createdtime >= '2024-02-01'
      AND total_area >= 18
),
calculated_prices AS (
    SELECT *,
           CASE
               WHEN price_uye <= 5000 THEN price_uye
               ELSE price_uye / total_area
           END AS price_per_m2_uye,
           CASE
               WHEN price_uz <= 5000 * {usd_to_uzs} THEN price_uz
               ELSE price_uz / total_area
           END AS price_per_m2_uz
    FROM converted_prices
),
filtered_data AS (
    SELECT *,
           CASE
               WHEN number_of_rooms = 1 THEN total_area BETWEEN 18 AND 70
               WHEN number_of_rooms = 2 THEN total_area BETWEEN 25 AND 100
               WHEN number_of_rooms = 3 THEN total_area BETWEEN 30 AND 150
               WHEN number_of_rooms = 4 THEN total_area BETWEEN 50 AND 150
               ELSE total_area BETWEEN 80 AND 400
           END AS valid_area
    FROM calculated_prices
)
SELECT regionname, number_of_rooms, total_area, floor, total_floors,
       furnished, repairs, comission, wc, house_type, isbusiness, ishighlighted, ispromoted, negotiable,
       layout, districtname, ceiling_height, price_per_m2_uye
FROM filtered_data
WHERE valid_area;
"""

# Загружаем данные
df = pd.read_sql(query, conn)
conn.close()

# Преобразуем ceiling_height в числовой формат
df["ceiling_height"] = pd.to_numeric(df["ceiling_height"], errors='coerce')

# Удаляем столбцы с пропусками ≥ 70%
df = df.loc[:, df.isnull().mean() < 0.7]

# Обрабатываем категориальные признаки (One-Hot Encoding)
df = pd.get_dummies(df, columns=["furnished", "repairs", "wc", "house_type", "layout"], drop_first=True)

# Заполняем пропуски средним значением для числовых признаков
df.fillna(df.mean(), inplace=True)

# Заменяем запись количества комнат на более понятную
room_labels = {
    1: "1-комнатные",
    2: "2-комнатные",
    3: "3-комнатные",
    4: "4-комнатные",
    5: "5 и более комнатные"
}

# Обновляем формирование таблицы результатов
results = []

for region, (lower_bound, upper_bound) in outlier_bounds.items():
    df_region = df[df["regionname"] == region]
    df_region = df_region[(df_region["price_per_m2_uye"] >= lower_bound) & (df_region["price_per_m2_uye"] <= upper_bound)]
    if df_region.empty:
        continue

    for rooms in room_labels.keys():
        df_filtered = df_region[df_region["number_of_rooms"] >= 5] if rooms == 5 else df_region[df_region["number_of_rooms"] == rooms]
        if df_filtered.empty:
            continue

        X = df_filtered.drop(columns=["price_per_m2_uye", "number_of_rooms", "regionname"])
        y = df_filtered["price_per_m2_uye"]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        model = xgb.XGBRegressor(objective="reg:squarederror", n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=1)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        mape = mean_absolute_percentage_error(y_test, y_pred)
        error_10_percent = (abs(y_test - y_pred) / y_test > 0.1).mean()

        results.append({
            "Регион": region,
            "Комнат": room_labels[rooms],
            "MAE": round(mae, 2),
            "MAPE (%)": round(mape * 100, 2),
            "Ошибки >10% (%)": round(error_10_percent * 100, 2)
        })

# Выводим таблицу
df_results = pd.DataFrame(results)
df_results


Текущий курс USD/UZS: 12879.95


/var/folders/q0/042269_x4qggn_0htwwx67qr0000gq/T/ipykernel_45694/1904177022.py:96: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,Регион,Комнат,MAE,MAPE (%),Ошибки >10% (%)
0,Бухарская область,1-комнатные,56.24,10.20,31.17
1,Бухарская область,2-комнатные,62.89,10.96,41.59
2,Бухарская область,3-комнатные,62.96,12.00,47.55
3,Бухарская область,4-комнатные,94.66,18.60,48.00
4,Бухарская область,5 и более комнатные,152.95,29.97,60.00
5,Самаркандская область,1-комнатные,157.36,16.67,45.83
6,Самаркандская область,2-комнатные,139.09,16.69,49.07
7,Самаркандская область,3-комнатные,120.58,16.56,48.05
8,Самаркандская область,4-комнатные,138.68,23.45,57.14
9,Самаркандская область,5 и более комнатные,187.00,27.79,72.73
